In [1]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.plotting import figure
from bokeh.io import show, output_notebook
import numpy as np
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
output_notebook()

Loading BokehJS ...

## Why Binning Helps Naive Bayes

Naive Bayes estimates class probabilities from feature occurrence counts. For continuous variables, using raw numeric values can create many unique tokens (for example, `age_RAW_37`, `age_RAW_38`, `age_RAW_39`) that each appear infrequently. With a small `max_features` budget, those sparse tokens compete for limited vocabulary space and can crowd out stronger signals.

Binning replaces nearby numeric values with broader categories (for example, `age_BIN_35_44`). This gives each feature more statistical support because more samples fall into the same bin. In practice, this usually makes probability estimates more stable, reduces sparsity, and helps the model prioritize informative patterns when only a small number of features is retained.

In [2]:
import importlib
import adult_preprocessing

importlib.reload(adult_preprocessing)
AdultIncomePreprocessor = adult_preprocessing.AdultIncomePreprocessor
AdultPreprocessConfig = adult_preprocessing.AdultPreprocessConfig

config = AdultPreprocessConfig(
    data_path="census+income/adult.data",
    continuous_mode="both",  # choose: "raw", "binned", or "both"
)
preprocessor = AdultIncomePreprocessor(config)

demographics, income = preprocessor.load()

print(f"Loaded {len(demographics)} rows from {config.data_path}")
print(f"Positive income count: {sum(income)}")
print(f"Continuous feature mode: {config.continuous_mode}")

print("\nBin frequency summary:")
for line in preprocessor.format_bin_frequency_summary():
    print(line)

Loaded 32561 rows from census+income/adult.data
Positive income count: 7841
Continuous feature mode: both

Bin frequency summary:
age bin frequencies (n=32561):
         25_34:   8479 (26.04%)
         35_44:   8151 (25.03%)
         45_54:   5853 (17.98%)
          lt25:   5570 (17.11%)
         55_64:   3172 ( 9.74%)
       65_plus:   1336 ( 4.10%)
capital_gain bin frequencies (n=32561):
          zero:  29849 (91.67%)
        1_4999:   1064 ( 3.27%)
     5000_9999:    878 ( 2.70%)
    10000_plus:    770 ( 2.36%)
capital_loss bin frequencies (n=32561):
          zero:  31042 (95.33%)
     1000_1999:   1158 ( 3.56%)
     2000_plus:    325 ( 1.00%)
         1_999:     36 ( 0.11%)
education_num bin frequencies (n=32561):
           mid:  20241 (62.16%)
          high:   7078 (21.74%)
           low:   4253 (13.06%)
     very_high:    989 ( 3.04%)
hours_per_week bin frequencies (n=32561):
         35_44:  18015 (55.33%)
         45_50:   5320 (16.34%)
         20_34:   3879 (11.91%)
    

In [3]:
train_demographics, test_demographics, train_income, test_income = train_test_split(demographics, income, random_state=11)
print('Length of train_demographics is ', len(train_demographics))
print('Length of test_demographics is ',len(test_demographics))

Length of train_demographics is  24420
Length of test_demographics is  8141


In [9]:
vectorizer = CountVectorizer(max_features=100,binary=True,stop_words='english')
V = vectorizer.fit(train_demographics)
X = V.transform(train_demographics)
train_y = np.array(train_income)
B = BernoulliNB().fit(X,train_y)

In [14]:
T = V.transform(test_demographics)
y_test = np.array(test_income)

In [15]:
print(vectorizer.get_feature_names_out())
print(V.vocabulary_)

['age_bin_25_34' 'age_bin_35_44' 'age_bin_45_54' 'age_bin_55_64'
 'age_bin_65_plus' 'age_bin_lt25' 'age_raw_23' 'age_raw_24' 'age_raw_25'
 'age_raw_27' 'age_raw_28' 'age_raw_29' 'age_raw_30' 'age_raw_31'
 'age_raw_32' 'age_raw_33' 'age_raw_34' 'age_raw_35' 'age_raw_36'
 'age_raw_37' 'age_raw_38' 'age_raw_39' 'age_raw_40' 'age_raw_42'
 'capital_gain_bin_10000_plus' 'capital_gain_bin_1_4999'
 'capital_gain_bin_5000_9999' 'capital_gain_bin_zero' 'capital_gain_raw_0'
 'capital_loss_bin_1000_1999' 'capital_loss_bin_zero' 'capital_loss_raw_0'
 'education_10th' 'education_11th' 'education_assoc_acdm'
 'education_assoc_voc' 'education_bachelors' 'education_hs_grad'
 'education_masters' 'education_num_bin_high' 'education_num_bin_low'
 'education_num_bin_mid' 'education_num_bin_very_high'
 'education_num_raw_10' 'education_num_raw_11' 'education_num_raw_12'
 'education_num_raw_13' 'education_num_raw_14' 'education_num_raw_6'
 'education_num_raw_7' 'education_num_raw_9' 'education_some_college'


In [16]:
y_pred = B.predict(T)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8080088441223436

Confusion matrix:
 [[5096 1155]
 [ 408 1482]]

Classification report:
               precision    recall  f1-score   support

           0       0.93      0.82      0.87      6251
           1       0.56      0.78      0.65      1890

    accuracy                           0.81      8141
   macro avg       0.74      0.80      0.76      8141
weighted avg       0.84      0.81      0.82      8141



In [19]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

# Export matrix for cross-model comparison plots.
os.makedirs("exports", exist_ok=True)
nb_cm_path = os.path.join("exports", "naive_bayes_confusion_matrix.csv")
np.savetxt(nb_cm_path, cm, fmt="%d", delimiter=",")
print(f"Saved Naive Bayes confusion matrix to: {nb_cm_path}")

Saved Naive Bayes confusion matrix to: exports\naive_bayes_confusion_matrix.csv
